# Recs 005: Structured game embeddings

## Key Goal

Generate structured-preference game embeddings and aligned artifacts for direct comparison against raw embeddings.

## Decision It Supports

Whether structured game indexing improves retrieval enough to justify extra preprocessing.

## Primary Metrics

Artifact parity and retrieval sanity checks versus raw index (coverage, dimensions, top-K behavior).


In [1]:
from __future__ import annotations

import sys
from pathlib import Path
import numpy as np
import pandas as pd

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root (no pyproject.toml). cwd={here}")

REPO_ROOT = _repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from steam_review_ml.recommender.retrieve import ContentRetriever
from steam_review_ml.recommender.preferences import extract_preferences, build_embedding_input

ARTIFACT_DIR = REPO_ROOT / "artifacts" / "recs"
REVIEWS_LONG_PATH = ARTIFACT_DIR / "game_profile_reviews.parquet"
OUT_NPZ = ARTIFACT_DIR / "game_profile_embeddings_structured_eval.npz"
OUT_INDEX = ARTIFACT_DIR / "game_profile_embedding_index_structured_eval.parquet"

MAX_REVIEWS_PER_GAME = 50

retriever = ContentRetriever()
print("Repo root:", REPO_ROOT)
print("Input:", REVIEWS_LONG_PATH)


Repo root: /home/ryanr/workspace/steam_recommendations
Input: /home/ryanr/workspace/steam_recommendations/artifacts/recs/game_profile_reviews.parquet


In [2]:
reviews_long = pd.read_parquet(REVIEWS_LONG_PATH)
reviews_long = reviews_long.dropna(subset=["review"]).copy()

if MAX_REVIEWS_PER_GAME is not None:
    reviews_long = (
        reviews_long.sort_values(["app_id", "review_id"], kind="mergesort")
        .groupby("app_id", as_index=False, sort=False)
        .head(MAX_REVIEWS_PER_GAME)
        .copy()
    )

print("Rows:", len(reviews_long))
print("Games:", reviews_long["app_id"].nunique())


Rows: 15672
Games: 315


In [3]:
def _l2_normalize(v: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = float(np.linalg.norm(v))
    if n <= eps:
        return v.astype(np.float32)
    return (v / n).astype(np.float32)

def _structured_text_from_review(review_text: str) -> str:
    raw = (review_text or "").strip()
    prefs = extract_preferences(raw)
    return build_embedding_input(prefs, raw)

rows = []
app_ids = []
names = []

for app_id, g in reviews_long.groupby("app_id", sort=True):
    structured_texts = [_structured_text_from_review(t) for t in g["review"].astype(str).tolist()]
    vecs = [retriever.embed_text(t) for t in structured_texts]
    m = np.mean(np.stack(vecs, axis=0), axis=0)
    rows.append(_l2_normalize(m))
    app_ids.append(int(app_id))
    names.append(str(g["app_name"].iloc[0]))

X_struct = np.stack(rows, axis=0).astype(np.float32)
app_ids_struct = np.asarray(app_ids, dtype=np.int64)
idx_struct = pd.DataFrame({"app_id": app_ids, "app_name": names})

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(OUT_NPZ, embeddings=X_struct, app_id=app_ids_struct)
idx_struct.to_parquet(OUT_INDEX, index=False)

print("Wrote:")
print("-", OUT_NPZ)
print("-", OUT_INDEX)
print("Structured matrix shape:", X_struct.shape)


2026-04-21 09:35:38.301798: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776778538.322052   61721 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776778538.329563   61721 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776778538.347163   61721 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776778538.347186   61721 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776778538.347188   61721 computation_placer.cc:177] computation placer alr

Wrote:
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/game_profile_embeddings_structured_eval.npz
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/game_profile_embedding_index_structured_eval.parquet
Structured matrix shape: (315, 512)
